### Read SLHA and SModelS output and store the data in a pandas DataFrame

In [1]:
import warnings
warnings.filterwarnings("ignore", message="numpy.dtype size changed")
import numpy as np
import pandas as pd
import glob,os,sys
import importlib
import importlib.util
from pandas import json_normalize
import pyslha
sys.path.append('../smodels')
pd.options.mode.chained_assignment = None #Disable copy warnings

In [2]:
slhaFolder = '../processFolders/Slhas/'
smodelsFolder = '../processFolders/smodelsOutput/'

In [3]:
slhaFolder = '../processFolders/Zp_SLHAs/'
smodelsFolder = '../processFolders/SmodelsOut_Zp/'

In [4]:
# Convert Experimental Results list to a dictionary
data = []
removeFromDict = ['topologies outside the grid', "missing topologies",
                  "missing topologies with displaced decays", 'missing topologies with prompt decays',
                  "Asymmetric Branches", "Outside Grid", "Missed Topologies", "Long Cascades"]

for f in glob.glob(smodelsFolder + '/*.py'):
    # --- Python 3.12+ replacement for imp.load_source ---
    module_name = os.path.basename(f).replace('.py', '')
    spec = importlib.util.spec_from_file_location(module_name, f)
    loaded_module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(loaded_module)
    
    # Extract the dictionary
    smodelsDict = loaded_module.smodelsOutput
    # ----------------------------------------------------
    
    # Remove unwanted keys
    for rmKey in removeFromDict:
        if rmKey in smodelsDict:
            smodelsDict.pop(rmKey)
            
    # Format the Experimental Results
    if 'ExptRes' in smodelsDict:
        for res in smodelsDict['ExptRes']:
            if 'TxNames weights (fb)' in res:
                res.pop('TxNames weights (fb)')  
                
        expList = sorted(smodelsDict['ExptRes'], key=lambda pt: pt['r'], reverse=True)
        expDict = dict([['result%i'%i, val] for i, val in enumerate(expList)])
        smodelsDict['ExptRes'] = expDict
        
    # Get the original SLHA filename and update the data dictionary
    slhaFile = smodelsDict['OutputStatus']['input file']
    dataDict = {'filename': os.path.basename(slhaFile)}
    dataDict.update(smodelsDict)
    
    # Append to the final data list
    data.append(dataDict)

In [5]:
print(len(data))

9269


In [6]:
#Convert data to flat DataFrame:
smodelsDF = json_normalize(data)

In [7]:
#Get SLHA data:
slhaData = []
for f in smodelsDF['filename']:
    slhaFile = os.path.join(slhaFolder,f)
    slha = pyslha.readSLHAFile(slhaFile)
    massDict = dict([[str(key),abs(val)] for key,val in slha.blocks['MASS'].items() if key > 25])
    widthDict = dict([[str(key),val.totalwidth] for key,val in slha.decays.items() if key > 25])
    #parsDict = dict([[str(key),abs(val)] for key,val in slha.blocks['FRBLOCK'].items()])
    parsDict = dict([[str(key),abs(val)] for key,val in slha.blocks['DMINPUTS'].items()])
    BRsDict = {}
    for pdg,val in slha.decays.items():
        if not abs(pdg) > 100000:
            continue
        initialState = str(pdg)
        BRsDict[initialState] = {}
        for dec in val.decays:
            if dec.br < 0.01: continue            
            finalState = ','.join([str(pid) for pid in sorted(dec.ids)])
            BRsDict[initialState][finalState] = dec.br
    xsec13TeV = dict([ [str(proc.pidsfinal).replace('[','').replace(']','').replace(',','_').replace(' ',''),
                   max([x.value for x in proc.get_xsecs(sqrts=13000)])*1000] 
                 for proc in slha.xsections.values()  if proc.get_xsecs(sqrts=13000)])    
    slhaDict = {'filename' : f, 'mass' : massDict, 'width' : widthDict, 
                'xsec13TeV(fb)' : xsec13TeV, 'BRs' : BRsDict, 'pars' : parsDict}
    slhaData.append(slhaDict)

In [8]:
#Convert to DataFrame
slhaDF = json_normalize(slhaData)
#Add total cross-sections:
xsecs13 = [x for x in list(slhaDF) if 'xsec13TeV' in x]
xsecs8 = [x for x in list(slhaDF) if 'xsec8TeV' in x]
slhaDF['totalxsec13TeV(fb)'] = slhaDF[xsecs13].sum(axis=1)
slhaDF['totalxsec8TeV(fb)'] = slhaDF[xsecs8].sum(axis=1)

In [9]:
#Merge with SModelS DataFrame
dataDF = slhaDF.merge(smodelsDF,how='inner')
# print('Final number of data points:',dataDF.shape[0])
#print(dataDF.columns.values.tolist()) #Print all columns names

In [10]:
#Save DataFrame to pickle file:

#dataDF.to_pickle('./VLF_smodels.pcl')
dataDF.to_pickle('./Zp_ttbar_smodels.pcl')

In [11]:
print(len(dataDF))

9269
